# Notebook 3: K-Means Clustering — Multiple Features + PCA

**Dataset:** Survey of Consumer Finances (SCF) 2019  
**Method:** K-Means clustering on 5 high-variance features, visualized with PCA

In this notebook we:
- Filter households with net worth < $2M and TURNFEAR == 1
- Compute (trimmed) variance to identify the top 5 features
- Scale features with StandardScaler
- Find optimal K using inertia and silhouette curves
- Reduce dimensions to 2D with PCA for visualization

## 0. Colab Setup

Run this cell first when working in Google Colab. It installs dependencies and mounts your Google Drive.

> **Before running:** Upload the `customer-segmentation/` folder to your Google Drive (or clone your GitHub repo).

In [ ]:
# ── Clone the GitHub repository ────────────────────────────────────────────
# Replace the URL below with your actual GitHub repo URL
REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO_NAME.git'

!git clone {REPO_URL}

# ── Set working directory to the project folder ────────────────────────────
import os

# Replace YOUR_REPO_NAME with the actual cloned folder name
os.chdir('/content/YOUR_REPO_NAME/customer-segmentation')

# ── Install required packages ──────────────────────────────────────────────
!pip install pandas numpy plotly scikit-learn scipy --quiet

print(f'Working directory: {os.getcwd()}')
print(f'Data file exists: {os.path.exists("data/SCF_2019.csv.gz")}')

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import mstats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '${:,.2f}'.format)

print('Libraries loaded!')

## 2. Wrangle Function

Filter to **TURNFEAR == 1** and **net worth < $2,000,000**.

In [ ]:
def wrangle(filepath):
    """
    Load and wrangle SCF 2019 data.
    
    Filters:
    - Implicate 1 only (every 5th row)
    - TURNFEAR == 1 (turned down or feared credit denial)
    - NETWORTH < 2,000,000 (focus on lower/middle wealth households)
    
    Parameters
    ----------
    filepath : str
        Path to compressed CSV (SCF_2019.csv.gz)
    
    Returns
    -------
    pd.DataFrame
    """
    df = pd.read_csv(filepath, compression='gzip', index_col=0)
    
    # Keep implicate 1
    df = df.iloc[::5].reset_index(drop=True)
    
    # Filter: TURNFEAR == 1
    df = df[df['TURNFEAR'] == 1].copy()
    
    # Filter: net worth < $2M
    df = df[df['NETWORTH'] < 2_000_000].copy()
    
    return df.reset_index(drop=True)


df = wrangle('data/SCF_2019.csv.gz')

print(f'Shape: {df.shape}')
print(f'Net worth range: ${df["NETWORTH"].min():,.0f} to ${df["NETWORTH"].max():,.0f}')
df.head()

## 3. Identify High-Variance Features

In [ ]:
# Select only numeric columns (exclude categorical/flag variables)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Exclude obvious non-feature columns
exclude = ['YY1', 'Y1', 'WGT', 'TURNFEAR', 'RACECL4', 'RACECL', 'HHSEX',
           'MARRIED', 'KIDS', 'OCCAT1', 'OCCAT2', 'EDUC', 'INCOME_RANK']
feature_cols = [c for c in numeric_cols if c not in exclude]

# Compute variance for all numeric features
variance_series = df[feature_cols].var().sort_values(ascending=False)

# Top 10 features by variance
top_ten_var = variance_series.head(10)

print('Top 10 features by variance:')
print(top_ten_var.to_string())

## 4. Bar Chart: High-Variance Features

In [ ]:
fig = px.bar(
    x=top_ten_var.values,
    y=top_ten_var.index,
    orientation='h',
    title='SCF: High Variance Features',
    labels={'x': 'Variance', 'y': 'Feature'},
    color=top_ten_var.values,
    color_continuous_scale='Blues'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## 5. Boxplot: Non-Home, Non-Financial Assets (NHNFIN)

In [ ]:
fig = px.box(
    df,
    x='NHNFIN',
    title='Distribution of Non-home, Non-Financial Assets',
    labels={'NHNFIN': 'Value [$]'},
    color_discrete_sequence=['#636EFA']
)
fig.show()

print(f'NHNFIN - Skewness: {df["NHNFIN"].skew():.2f}')
print('(values > 1.0 indicate right-skewed distribution — extreme high values)')

## 6. Trimmed Variance

Since the raw data is highly skewed, we use **trimmed variance** (removing extreme values before computing variance) to identify features that truly vary across typical households.

In [ ]:
def trimmed_variance(x, proportiontocut=0.1):
    """
    Compute variance after trimming the top and bottom proportiontocut
    of values from a series.
    
    Parameters
    ----------
    x : array-like
    proportiontocut : float (default 0.1 = 10% from each tail)
    
    Returns
    -------
    float : trimmed variance
    """
    x_trimmed = mstats.trimr(x, proportiontocut, proportiontocut)
    return np.var(x_trimmed.compressed())


# Compute trimmed variance for all features
trim_var_values = {
    col: trimmed_variance(df[col].dropna().values)
    for col in feature_cols
    if df[col].notna().sum() > 100  # need enough data
}
trim_var_series = pd.Series(trim_var_values).sort_values(ascending=False)

# Top 10 by trimmed variance
top_ten_trim_var = trim_var_series.head(10)

print('Top 10 features by TRIMMED variance:')
print(top_ten_trim_var.to_string())

## 7. Bar Chart: Trimmed High-Variance Features

In [ ]:
fig = px.bar(
    x=top_ten_trim_var.values,
    y=top_ten_trim_var.index,
    orientation='h',
    title='SCF: High Trimmed-Variance Features',
    labels={'x': 'Trimmed Variance', 'y': 'Feature'},
    color=top_ten_trim_var.values,
    color_continuous_scale='Oranges'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

## 8. Select Top 5 High-Variance Columns

In [ ]:
# Top 5 features with highest trimmed variance
high_var_cols = top_ten_trim_var.head(5).index.tolist()

print('Top 5 high-variance features selected for clustering:')
for i, col in enumerate(high_var_cols, 1):
    print(f'  {i}. {col}')

## 9. Feature Matrix and Summary Statistics

In [ ]:
# Create feature matrix
X = df[high_var_cols].copy()

print(f'Feature matrix shape: {X.shape}')

# Summary statistics before scaling
X_summary = pd.DataFrame({
    'Mean': X.mean(),
    'Std Dev': X.std()
})

print('\nX Summary (before scaling):')
print(X_summary.to_string())

## 10. StandardScaler: Normalize Features

In [ ]:
# Create and fit StandardScaler
scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X)

# Put into DataFrame
X_scaled = pd.DataFrame(X_scaled_array, columns=high_var_cols)

# Summary after scaling
X_scaled_summary = pd.DataFrame({
    'Mean': X_scaled.mean(),
    'Std Dev': X_scaled.std()
})

print('X_scaled Summary (after StandardScaler):')
print(X_scaled_summary.round(6).to_string())
print('\n(Mean ≈ 0, Std Dev ≈ 1 after scaling)')

## 11. K-Means Pipeline: Evaluate K from 2 to 12

In [ ]:
n_clusters_range = range(2, 13)
inertia_errors = []
silhouette_scores = []

for k in n_clusters_range:
    # Pipeline: StandardScaler + KMeans
    pipeline = make_pipeline(
        StandardScaler(),
        KMeans(n_clusters=k, random_state=42, n_init=10)
    )
    pipeline.fit(X)
    
    # Inertia from the KMeans step
    inertia = pipeline.named_steps['kmeans'].inertia_
    inertia_errors.append(inertia)
    
    # Silhouette score (on scaled data)
    labels_k = pipeline.named_steps['kmeans'].labels_
    X_scaled_temp = pipeline.named_steps['standardscaler'].transform(X)
    sil = silhouette_score(X_scaled_temp, labels_k)
    silhouette_scores.append(sil)
    
    print(f'K={k:2d} | Inertia: {inertia:>15,.2f} | Silhouette: {sil:.4f}')

print('\nEvaluation complete!')

## 12. Inertia vs. Number of Clusters

In [ ]:
fig = px.line(
    x=list(n_clusters_range),
    y=inertia_errors,
    markers=True,
    title='K-Means Model: Inertia vs Number of Clusters',
    labels={'x': 'Number of Clusters', 'y': 'Inertia'}
)
fig.update_traces(line_color='#636EFA', marker_color='#EF553B', marker_size=9)
fig.update_layout(xaxis=dict(tickmode='linear', tick0=2, dtick=1))
fig.show()

## 13. Silhouette Score vs. Number of Clusters

In [ ]:
fig = px.line(
    x=list(n_clusters_range),
    y=silhouette_scores,
    markers=True,
    title='K-Means Model: Silhouette Score vs Number of Clusters',
    labels={'x': 'Number of Clusters', 'y': 'Silhouette Score'}
)
fig.update_traces(line_color='#00CC96', marker_color='#EF553B', marker_size=9)
fig.update_layout(xaxis=dict(tickmode='linear', tick0=2, dtick=1))
fig.show()

best_k_idx = np.argmax(silhouette_scores)
best_k = list(n_clusters_range)[best_k_idx]
print(f'Best K by silhouette: K={best_k} (score={silhouette_scores[best_k_idx]:.4f})')

## 14. Final Model

In [ ]:
# Build final model — adjust BEST_K based on your plots
BEST_K = best_k

final_model = make_pipeline(
    StandardScaler(),
    KMeans(n_clusters=BEST_K, random_state=42, n_init=10)
)
final_model.fit(X)

# Extract labels
labels = final_model.named_steps['kmeans'].labels_
final_inertia = final_model.named_steps['kmeans'].inertia_
X_scaled_final = final_model.named_steps['standardscaler'].transform(X)
final_sil = silhouette_score(X_scaled_final, labels)

print(f'Final Model (K={BEST_K})')
print(f'  Inertia:          {final_inertia:,.2f}')
print(f'  Silhouette Score: {final_sil:.4f}')

# Cluster sizes
for c, cnt in pd.Series(labels).value_counts().sort_index().items():
    print(f'  Cluster {c}: {cnt:,} ({cnt/len(labels)*100:.1f}%)')

## 15. Mean Household Finances by Cluster

In [ ]:
# DataFrame of mean values per cluster
xgb = X.copy()
xgb['Cluster'] = labels
xgb = xgb.groupby('Cluster')[high_var_cols].mean().reset_index()
xgb['Cluster'] = xgb['Cluster'].astype(str)

print('Mean feature values per cluster:')
print(xgb.to_string(index=False))

# Melt for grouped bar chart
xgb_melted = xgb.melt(id_vars='Cluster', var_name='Feature', value_name='Mean Value ($)')

fig = px.bar(
    xgb_melted,
    x='Cluster',
    y='Mean Value ($)',
    color='Feature',
    barmode='group',
    title='Mean Household Finances by Cluster',
    labels={'Cluster': 'Cluster', 'Mean Value ($)': 'Value [$]'},
    color_discrete_sequence=px.colors.qualitative.Plotly
)
fig.show()

## 16. PCA: Reduce to 2 Dimensions

In [ ]:
# Create PCA transformer and reduce to 2 components
pca = PCA(n_components=2, random_state=42)
X_pca_array = pca.fit_transform(X_scaled_final)

X_pca = pd.DataFrame(X_pca_array, columns=['PC1', 'PC2'])

print(f'PCA explained variance ratio:')
print(f'  PC1: {pca.explained_variance_ratio_[0]*100:.1f}%')
print(f'  PC2: {pca.explained_variance_ratio_[1]*100:.1f}%')
print(f'  Total: {sum(pca.explained_variance_ratio_)*100:.1f}%')

X_pca.head()

## 17. PCA Scatter Plot of Clusters

In [ ]:
X_pca['labels'] = labels.astype(str)

fig = px.scatter(
    X_pca,
    x='PC1',
    y='PC2',
    color='labels',
    title='PCA Representation of Clusters',
    labels={'PC1': 'PC1', 'PC2': 'PC2', 'labels': 'Cluster'},
    opacity=0.6,
    color_discrete_sequence=px.colors.qualitative.Plotly
)
fig.update_layout(
    annotations=[
        dict(
            text=f'Total variance explained: {sum(pca.explained_variance_ratio_)*100:.1f}%',
            xref='paper', yref='paper',
            x=0.01, y=0.99,
            showarrow=False,
            bgcolor='white',
            bordercolor='gray',
            borderwidth=1
        )
    ]
)
fig.show()

## 18. Cluster Profile Summary

In [ ]:
# Add cluster labels to original df for profiling
df_clustered = df.copy()
df_clustered['Cluster'] = labels

profile_cols = ['INCOME', 'NETWORTH', 'ASSET', 'DEBT', 'AGE'] + high_var_cols
profile_cols = list(dict.fromkeys(profile_cols))  # deduplicate

profile = df_clustered.groupby('Cluster')[profile_cols].agg(['mean', 'median'])
profile.columns = ['_'.join(col) for col in profile.columns]

print('Cluster Profiles (mean | median):')
print(profile.T.to_string())